# 👁️ AI VISION ASSISTANT: Huấn luyện QLoRA cho Qwen2-VL-2B trên Google Colab
**Mô tả:** Notebook mẫu hoàn chỉnh để fine-tuning mô hình Vision-Language (Qwen2-VL-2B-Instruct) trên GPU miễn phí của Google Colab (Tesla T4 15GB VRAM) sử dụng kỹ thuật QLoRA 4-bit.
Sau khi hoàn tất, adapter sẽ được nén và tải về để đặt vào thư mục `adapters/` của dự án local.

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi
import torch
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q -U torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate peft bitsandbytes datasets qwen-vl-utils evaluate sacrebleu rouge-score

In [ ]:
# 3. Tải và chuẩn bị bộ dữ liệu OpenViVQA
from datasets import load_dataset
print("Đang nạp dataset OpenViVQA...")
dataset = load_dataset("uitnlp/OpenViVQA-dataset", trust_remote_code=True)
print(dataset)

In [ ]:
# 4. Khởi tạo Processor và Model Qwen2-VL-2B với lượng tử hóa 4-bit NF4
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig

model_id = "Qwen/Qwen2-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Đã nạp Base Model thành công!")

In [ ]:
# 5. Cấu hình LoRA Adapter và Freeze Vision Tower
from peft import LoraConfig, get_peft_model, TaskType

# Đóng băng Vision Encoder để tiết kiệm VRAM
for name, param in model.visual.named_parameters():
    param.requires_grad = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type=TaskType.CAUSAL_LM
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

In [ ]:
# 6. Thiết lập Training Arguments và Bắt đầu Huấn luyện
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./checkpoints/qwen_colab",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none"
)

print("Cấu hình huấn luyện sẵn sàng! Bạn có thể truyền dataset và khởi chạy trainer.train().")

In [ ]:
# 7. Xuất Adapter và Tải về Máy Cục bộ
output_adapter_dir = "./qwen_vietnamese_qlora"
peft_model.save_pretrained(output_adapter_dir)
print(f"Đã lưu adapter tại {output_adapter_dir}")

# Nén thành zip để tải về
!zip -r qwen_vietnamese_qlora.zip ./qwen_vietnamese_qlora

from google.colab import files
files.download('qwen_vietnamese_qlora.zip')
print("Tải file zip về, giải nén và đặt vào thư mục AI_VISION_ASSISTANT/adapters/")